# GeoFireX Certify lot 004 CPU 20261108 V1
Prepared locally; never executed by preparation.


In [ ]:
import base64,io,json,os,sys,zipfile,zlib
from pathlib import Path
os.environ.update(CUDA_VISIBLE_DEVICES='',OMP_NUM_THREADS='2',MKL_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2')
work=Path('/kaggle/working/lot-004-certification')
work.mkdir(exist_ok=False)
archive=zipfile.ZipFile(io.BytesIO(base64.b64decode('...')))
archive.extractall(work/'source')
sys.path.insert(0,str(work/'source'))
helpers={'__name__':'cpu_helpers'}
exec(compile('"""CPU-only Kaggle input helpers; bounded streaming, no network or training."""\nimport hashlib\nimport importlib\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport shutil\nimport sys\nimport time\nimport zipfile\nimport zlib\n\nBLOCK = 1024*1024\n\n\ndef sha_file(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(BLOCK), b\'\'): h.update(block)\n    return h.hexdigest()\n\n\ndef safe_child(root, name):\n    p = PurePosixPath(name)\n    if p.is_absolute() or \'..\' in p.parts or \'\\\\\' in name or \':\' in name:\n        raise ValueError(\'Unsafe relative path\')\n    target = Path(root)/name\n    if not target.resolve().is_relative_to(Path(root).resolve()):\n        raise ValueError(\'Path escapes root\')\n    return target\n\n\ndef bootstrap_code(source, work, payload_sha, manifest_sha):\n    source, work = Path(source), Path(work)\n    if work.exists(): raise FileExistsError(\'New working directory required\')\n    payload = source/\'payload.zip\'\n    archive = None\n    try:\n        if payload.is_file():\n            if sha_file(payload) != payload_sha: raise ValueError(\'Code ZIP SHA mismatch\')\n            archive = zipfile.ZipFile(payload)\n            names = archive.namelist()\n            manifest_bytes = archive.read(\'code-manifest.json\')\n            manifest = json.loads(manifest_bytes)\n            if len(names) != len(set(names)) or set(names) != set(manifest) | {\'code-manifest.json\'}:\n                raise ValueError(\'Code ZIP inventory differs\')\n        else:\n            matches = list(source.rglob(\'code-manifest.json\'))\n            if len(matches) != 1: raise ValueError(\'Expected one expanded code manifest\')\n            source = matches[0].parent\n            manifest_bytes = matches[0].read_bytes()\n            manifest = json.loads(manifest_bytes)\n        if hashlib.sha256(manifest_bytes).hexdigest() != manifest_sha:\n            raise ValueError(\'Pinned code manifest SHA mismatch\')\n        work.mkdir(parents=True, exist_ok=False)\n        for name, sha in manifest.items():\n            dest = safe_child(work, name)\n            origin = None if archive else safe_child(source, name)\n            if origin and origin.is_symlink(): raise ValueError(\'Symlink refused\')\n            dest.parent.mkdir(parents=True, exist_ok=True)\n            h = hashlib.sha256()\n            with (archive.open(name) if archive else origin.open(\'rb\')) as inp, dest.open(\'xb\') as out:\n                for block in iter(lambda: inp.read(BLOCK), b\'\'): h.update(block); out.write(block)\n            if h.hexdigest() != sha: raise ValueError(\'Expanded code file SHA mismatch: \'+name)\n        (work/\'code-manifest.json\').write_bytes(manifest_bytes)\n        return work\n    finally:\n        if archive: archive.close()\n\n\ndef check_runtime(work, bundle_relative):\n    if sys.version_info < (3, 11): raise RuntimeError(\'Operational preflight requires Python >=3.11\')\n    os.environ.update(CUDA_VISIBLE_DEVICES=\'\', OMP_NUM_THREADS=\'2\', MKL_NUM_THREADS=\'2\', OPENBLAS_NUM_THREADS=\'2\')\n    sys.path.insert(0, str(Path(work)/\'src\'))\n    versions, missing = {}, []\n    for module, distribution in [(\'numpy\',\'numpy\'), (\'rasterio\',\'rasterio\'), (\'torch\',\'torch\'), (\'mlflow\',\'mlflow\'),\n                                 (\'yaml\',\'pyyaml\'), (\'matplotlib\',\'matplotlib\'), (\'sklearn\',\'scikit-learn\'),\n                                 (\'pyarrow\',\'pyarrow\'), (\'psutil\',\'psutil\')]:\n        try:\n            importlib.import_module(module)\n            versions[distribution] = importlib.metadata.version(distribution)\n        except ImportError: missing.append(distribution)\n    if missing: raise RuntimeError(\'Missing dependencies; STOP, no automatic installation: \'+\',\'.join(missing))\n    psutil_version = tuple(int(v) for v in versions[\'psutil\'].split(\'.\')[:2])\n    if not (psutil_version >= (5,9) and psutil_version < (8,0)):\n        raise RuntimeError(\'psutil outside reviewed >=5.9,<8 range\')\n    import pyarrow.parquet as pq\n    bundle = Path(work)/bundle_relative\n    parquet_rows = {name: pq.read_table(bundle/f\'metadata/{name}.parquet\').num_rows for name in (\'observations\',\'windows\')}\n    norm_path = bundle/\'metadata/normalization.json\'\n    norm = json.loads(norm_path.read_text())\n    if sha_file(norm_path) != \'aa30df4e3d6d895760f04e35d74bcbbac0d739cea045d236517401f5f9f4c7f6\':\n        raise ValueError(\'Train normalizer SHA mismatch\')\n    provenance = norm[\'provenance\']\n    if (provenance[\'train_events\'], provenance[\'validation_inputs\'], provenance[\'test_inputs\']) != (250,0,0):\n        raise ValueError(\'Train-only provenance differs\')\n    working_free=shutil.disk_usage(\'/kaggle/working\').free\n    scratch=Path(\'/kaggle/temp\'); scratch.mkdir(exist_ok=True)\n    scratch_free=shutil.disk_usage(scratch).free\n    return dict(python=sys.version, versions=versions, parquet_rows=parquet_rows,\n                normalizer_sha256=sha_file(norm_path), train_events=250, validation_fit_inputs=0, test_fit_inputs=0,\n                working_free_bytes=working_free, scratch_free_bytes=scratch_free,\n                disk_gate_for_future_training=working_free >= 20_000_000_000 and scratch_free >= 40_000_000_000)\n\n\ndef resource_check(deadline, min_available=2*1024**3):\n    if time.monotonic() > deadline: raise TimeoutError(\'CPU certification deadline reached\')\n    mem = Path(\'/proc/meminfo\')\n    if mem.exists():\n        available = next(int(row.split()[1])*1024 for row in mem.read_text().splitlines() if row.startswith(\'MemAvailable:\'))\n        if available < min_available: raise MemoryError(\'CPU certification memory reserve reached\')\n\n\ndef certify_lot(canonical_path, dataset_root, out, *, pinned_identity, seconds=3600, inspect_geotiff=True):\n    lot = json.loads(Path(canonical_path).read_text())\n    unsigned = {k: v for k, v in lot.items() if k != \'identity\'}\n    identity = hashlib.sha256(json.dumps(unsigned, sort_keys=True, separators=(\',\', \':\')).encode()).hexdigest()\n    if identity != lot[\'identity\'] or identity != pinned_identity or lot[\'role\'] not in (\'train\',\'val\'):\n        raise ValueError(\'Pinned lot identity/role mismatch\')\n    dataset_root, out = Path(dataset_root), Path(out)\n    if json.loads((dataset_root/\'lot-inventory.json\').read_text()) != lot:\n        raise ValueError(\'Published inventory differs from pinned canonical manifest\')\n    expected = {f[\'path\'] for f in lot[\'files\']}\n    if len(expected) != lot[\'observations\'] or sum(f[\'bytes\'] for f in lot[\'files\']) != lot[\'bytes\']:\n        raise ValueError(\'Manifest counts differ\')\n    if {str(PurePosixPath(p).parent) for p in expected} != set(lot[\'events\']):\n        raise ValueError(\'Manifest event set differs\')\n    for p in expected:\n        if PurePosixPath(p).parts[0] not in (\'2018\',\'2019\',\'2020\'): raise ValueError(\'Held-out year refused\')\n    rasters = dataset_root/\'rasters\'\n    actual = {p.relative_to(rasters).as_posix() for p in rasters.rglob(\'*\') if p.is_file()}\n    if actual != expected: raise ValueError(\'Published raster inventory differs\')\n    out.mkdir(parents=True, exist_ok=False)\n    deadline, checked_bytes = time.monotonic()+seconds, 0\n    with (out/\'verified.jsonl\').open(\'x\', encoding=\'utf-8\') as log:\n        for f in lot[\'files\']:\n            resource_check(deadline)\n            path = safe_child(rasters, f[\'path\'])\n            if path.is_symlink() or path.stat().st_size != f[\'bytes\']: raise ValueError(\'Published size/path differs\')\n            sha, crc, n = hashlib.sha256(), 0, 0\n            with path.open(\'rb\') as fp:\n                for block in iter(lambda: fp.read(BLOCK), b\'\'):\n                    resource_check(deadline)\n                    sha.update(block); crc=zlib.crc32(block,crc); n+=len(block)\n            if n != f[\'bytes\'] or sha.hexdigest() != f[\'sha256\'] or f\'{crc & 0xffffffff:08x}\' != f[\'crc32\']:\n                raise ValueError(\'Published SHA256/CRC differs: \'+f[\'path\'])\n            checked_bytes += n\n            log.write(json.dumps(dict(path=f[\'path\'], sha256=sha.hexdigest(), bytes=n,\n                                      crc32=f\'{crc & 0xffffffff:08x}\'))+\'\\n\'); log.flush()\n    geotiff = None\n    if inspect_geotiff:\n        import rasterio\n        from rasterio.windows import Window\n        with rasterio.Env(GDAL_CACHEMAX=64*1024**2, GDAL_NUM_THREADS=\'1\'):\n            with rasterio.open(safe_child(rasters,lot[\'files\'][0][\'path\'])) as src:\n                tile = src.read(1, window=Window(0,0,min(64,src.width),min(64,src.height)), masked=True)\n                if tile.size == 0: raise ValueError(\'Empty GeoTIFF window\')\n                geotiff = dict(bands=src.count, width=src.width, height=src.height, tile_shape=list(tile.shape), crs=str(src.crs))\n    result = dict(status=\'ONE_LOT_REMOTE_BYTES_CERTIFIED\', handle=lot[\'handle\'], identity=identity,\n                  files=len(expected), events=len(lot[\'events\']), bytes=checked_bytes, role=lot[\'role\'], geotiff=geotiff,\n                  claim=\'This receipt certifies only this attached lot; no training or other lots were validated\')\n    with (out/\'certificate.json\').open(\'x\',encoding=\'utf-8\') as f: json.dump(result,f,indent=2)\n    return result\n','kaggle_certify_inputs.py','exec'),helpers)
namespace={'__name__':'cpu_lot_runtime'}
exec(compile('"""Private CPU certification of one frozen lot 002..008; no network or training."""\nimport datetime as dt\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport threading\nimport time\n\n\ndef run(reference, observations, normalization, split, helpers, output):\n    output=Path(output)\n    output.mkdir(parents=True,exist_ok=False)\n    started=time.monotonic()\n    deadline=started+1800\n    import psutil\n    proc=psutil.Process()\n    stop=threading.Event()\n    resources=dict(peak_rss_bytes=0,minimum_available_bytes=psutil.virtual_memory().available)\n    def guard():\n        rss=proc.memory_info().rss\n        available=psutil.virtual_memory().available\n        resources[\'peak_rss_bytes\']=max(resources[\'peak_rss_bytes\'],rss)\n        resources[\'minimum_available_bytes\']=min(resources[\'minimum_available_bytes\'],available)\n        if time.monotonic()>deadline: raise TimeoutError(\'1800 second certification deadline\')\n        if rss>3*1024**3 or available<2*1024**3: raise MemoryError(\'CPU certification reserve/limit reached\')\n    def watch():\n        while not stop.wait(1):\n            try: guard()\n            except Exception as error:\n                (output/\'resource-stop.json\').write_text(json.dumps(dict(error=str(error),**resources)))\n                os._exit(75)\n    watcher=threading.Thread(target=watch,daemon=True)\n    watcher.start()\n    try:\n        guard()\n        if reference[\'number\'] not in range(2,9) or reference[\'role\'] not in (\'train\',\'val\'):\n            raise ValueError(\'Only frozen lots 002..008 allowed; lot 001 is closed\')\n        if os.environ.get(\'KAGGLE_ACCELERATOR_TYPE\',\'\').lower() not in (\'\',\'none\',\'cpu\') or list(Path(\'/dev\').glob(\'nvidia[0-9]*\')):\n            raise RuntimeError(\'CPU-only session required\')\n        os.environ.update(CUDA_VISIBLE_DEVICES=\'\',OMP_NUM_THREADS=\'2\',MKL_NUM_THREADS=\'2\',OPENBLAS_NUM_THREADS=\'2\')\n        role=reference[\'role\']\n        permitted=(2018,2019) if role==\'train\' else (2020,)\n        if any(split.get(e)!=role or int(e.split(\'/\')[0]) not in permitted for e in reference[\'events\']):\n            raise ValueError(\'Event role/year mismatch\')\n        for f in reference[\'files\']:\n            if dt.date.fromisoformat(PurePosixPath(f[\'path\']).stem).year not in permitted:\n                raise ValueError(\'Observation outside lot role years: \'+f[\'path\'])\n        inventories=list(Path(\'/kaggle/input\').rglob(\'lot-inventory.json\'))\n        if len(inventories)!=1: raise ValueError(\'Exactly one data lot must be attached\')\n        root=inventories[0].parent\n        attached=[p for p in Path(\'/kaggle/input\').rglob(\'*\') if p.is_file()]\n        if len(attached)!=reference[\'observations\']+1 or any(\'2021\' in p.parts or p.stem.startswith(\'2021-\') for p in attached):\n            raise ValueError(\'Unexpected attached files or held-out 2021 data\')\n        canonical=output/\'canonical.json\'\n        canonical.write_text(json.dumps(reference))\n        proof=helpers[\'certify_lot\'](canonical,root,output/\'hashes\',pinned_identity=reference[\'identity\'],seconds=1700,inspect_geotiff=False)\n        guard()\n        import numpy as np\n        import rasterio\n        import torch\n        from rasterio.windows import Window\n        from torch.utils.data import DataLoader\n        from geofirex.unet_data import Normalizer,validate_layout\n        from geofirex.benchmark import aligned_windows\n        from geofirex.runner_data import FoldData,TemporalStore,TrainTiles\n        torch.set_num_threads(2)\n        torch.set_num_interop_threads(1)\n        rasters=root/\'rasters\'\n        resolver=FoldData.__new__(FoldData)\n        resolver.config={\'corpus_scope\':\'train_validation\'}\n        resolver.root=rasters.resolve()\n        resolver.roots=[rasters.resolve()]\n        expected={f[\'path\'] for f in reference[\'files\']}\n        if {r[\'relative_path\'] for r in observations}!=expected: raise ValueError(\'QC observation coverage mismatch\')\n        for name in expected:\n            if resolver.resolve_path(name)!=(rasters/name).resolve(): raise ValueError(\'Loader path resolution mismatch\')\n        samples=[]\n        loader_reads=[]\n        with rasterio.Env(GDAL_CACHEMAX=64*1024**2,GDAL_NUM_THREADS=\'1\'):\n            for i in (0,len(reference[\'files\'])//2,-1):\n                guard()\n                name=reference[\'files\'][i][\'path\']\n                with rasterio.open(rasters/name) as src:\n                    validate_layout(src)\n                    a=src.read(window=Window(0,0,min(64,src.width),min(64,src.height)))\n                    if src.crs is None or a.shape[0]!=23 or np.isinf(a).any(): raise ValueError(\'GeoTIFF layout/read invalid\')\n                    samples.append(dict(path=name,bands=src.count,width=src.width,height=src.height,crs=str(src.crs),tile_shape=list(a.shape)))\n            relocated=[{**r,\'relative_path\':str(resolver.resolve_path(r[\'relative_path\']))} for r in observations]\n            groups,rejected=aligned_windows(relocated,split,role,contexts=(1,3,5))\n            norm=Normalizer(**normalization[\'normalization\'])\n            for context in (1,3,5):\n                guard()\n                if not groups[context]: raise ValueError(\'No valid temporal window\')\n                window=groups[context][0]\n                maximum=groups[5][0]\n                geometry=None\n                for p in (*maximum.context_paths,maximum.target_path):\n                    with rasterio.open(p) as src:\n                        validate_layout(src)\n                        current=(src.shape,str(src.crs),tuple(src.transform))\n                        if geometry is not None and current!=geometry: raise ValueError(\'Temporal geometry mismatch\')\n                        geometry=current\n                store=TemporalStore([window],norm,domain_windows=[maximum],cache_bytes=0)\n                dataset=TrainTiles(store,size=64,samples=1,seed=42)\n                x,y,valid=next(iter(DataLoader(dataset,batch_size=1,num_workers=0,pin_memory=False)))\n                if list(x.shape)!=[1,context,8,64,64] or list(y.shape)!=[1,1,64,64]: raise ValueError(\'Loader shape mismatch\')\n                if any(t.device.type!=\'cpu\' or not torch.isfinite(t).all() for t in (x,y,valid)): raise ValueError(\'Loader device/finite mismatch\')\n                loader_reads.append(dict(context_days=context,X=list(x.shape),Y=list(y.shape),device=str(x.device),finite=True))\n        proof.update(status=\'LOT_REMOTE_CPU_CERTIFIED\',number=reference[\'number\'],anomalies=[],gpu_used=False,training=False,\n            rasterio_reads=samples,loader_reads=loader_reads,aligned_windows={str(k):len(v) for k,v in groups.items()},\n            window_rejections=rejected,test_events=0,test_2021_files=0,data_root=str(rasters),windows_drive_dependency=False,\n            normalizer_reused_without_fitting=True,resources=dict(**resources,elapsed_seconds=time.monotonic()-started))\n        (output/\'certificate.json\').write_text(json.dumps(proof,indent=2))\n        (output/\'verified.jsonl\').write_bytes((output/\'hashes/verified.jsonl\').read_bytes())\n        (output/\'progress.json\').write_text(json.dumps(dict(status=\'COMPLETE\',files_certified=proof[\'files\'],bytes_certified=proof[\'bytes\'])))\n        print(json.dumps(proof,indent=2),flush=True)\n    except Exception as error:\n        (output/\'failure.json\').write_text(json.dumps(dict(status=\'FAILED_STOP\',error_type=type(error).__name__,error=str(error))))\n        raise\n    finally:\n        stop.set()\n        watcher.join(timeout=2)\n','certify_kaggle_planned_lot_runtime.py','exec'),namespace)
payload=json.loads(zlib.decompress(base64.b64decode('...')))
namespace['run'](**payload,helpers=helpers,output=work/'results')
